In [ ]:
pip install kagglehub

: 

In [ ]:
import kagglehub
import pandas as pd
import os
import glob
import shutil
import numpy as np

In [ ]:
data_dir = "Data/Raw_Data"
#creates dir if not found
os.makedirs(data_dir, exist_ok=True)

dfs = []

if len(os.listdir(data_dir)) > 0:
    print("Data found locally. Skipping download.")
else:
    download_path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
    files = glob.glob(os.path.join(download_path, "**", "*.csv"), recursive=True)

    print(f"Found {len(files)} CSV files.")

    for file in files:
        print(f"Reading: {file}")
        try:
            df = pd.read_csv(file)
            dfs.append(df)
            shutil.copy2(file, data_dir)
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if dfs:
        data = pd.concat(dfs, ignore_index=True)
        output_path = os.path.join(data_dir, "combined_raw_data.csv")
        data.to_csv(output_path, index=False)
        print("Success! Data saved in {data_dir}")
    else:
        print("Failed to download and combine.")

In [ ]:
data = pd.read_csv("Data/Raw_Data/combined_raw_data.csv")

In [ ]:
data.head(10)

In [ ]:
data.columns

In [ ]:
data.drop(columns=[' Packet Length Variance', ' Fwd Header Length.1', ' Bwd PSH Flags', 'Fwd PSH Flags',
                   ' Fwd URG Flags',' Bwd URG Flags','FIN Flag Count',' SYN Flag Count',
                   ' RST Flag Count',' PSH Flag Count',' ACK Flag Count',' URG Flag Count',
                   ' CWE Flag Count',' ECE Flag Count','Fwd Avg Bytes/Bulk',' Fwd Avg Packets/Bulk',
                   ' Fwd Avg Bulk Rate',' Bwd Avg Bytes/Bulk',' Bwd Avg Packets/Bulk','Bwd Avg Bulk Rate',
                   'Subflow Fwd Packets',' Subflow Fwd Bytes',' Subflow Bwd Packets',' Subflow Bwd Bytes',
                   'Init_Win_bytes_forward',' Init_Win_bytes_backward',' act_data_pkt_fwd',' min_seg_size_forward',
                   ' Active Std',' Active Max',' Active Min',' Idle Std',' Idle Max',' Idle Min', ' Fwd IAT Min',
                   'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
                   ' Bwd IAT Max', ' Bwd IAT Min', ' Fwd Header Length',
                   ' Bwd Header Length', ' Down/Up Ratio', ' Avg Fwd Segment Size',
                   ' Avg Bwd Segment Size', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', 'Bwd Packet Length Max',
                   ' Bwd Packet Length Min', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Min Packet Length',
                   ' Max Packet Length', 'Active Mean', 'Idle Mean', ' Destination Port', ' Fwd Packet Length Std',  ' Bwd Packet Length Std',
                   'Fwd Packets/s', ' Bwd Packets/s'], inplace=True)

data.columns

In [ ]:
data.columns = data.columns.str.strip()
data.columns

In [ ]:
data = data.convert_dtypes()
data.dtypes

In [ ]:
data.isna().sum()

In [ ]:
data.isnull().sum()

In [ ]:
data.isnull().sum().sum()
#data.isnull().values.any()

In [ ]:
data.isna().sum().sum()

In [ ]:
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True) ###remove

In [ ]:
data.isna().sum().sum()
#All NaN/+/-inf is removed

In [ ]:
data.isnull().sum().sum()
#All NaN/+/-inf is removed

In [ ]:
data.drop_duplicates(inplace=True)
data.duplicated().sum()
#Removing all duplicates while  keeping one copy

In [ ]:
data['Label'].value_counts()

In [ ]:
data['Label'] = data['Label'].astype(str).str.strip()
data['Label'] = data['Label'].apply(
    lambda x: 'Benign' if x.upper() == 'BENIGN' else 'Malicious'
)

In [ ]:
data['Label'].value_counts()
#having only 'benign' and 'malicious as labels'

In [ ]:
data.head(2)

In [ ]:
#pd.options.display.float_format = '{:5d}'.format

In [ ]:
from sklearn.utils import resample
#SAMPLING DATA

# Separate by label
data_benign = data[data['Label'] == 'Benign']
data_malicious = data[data['Label'] == 'Malicious']

# Downsample Benign Label
data_benign_downsampled = resample(data_benign, 
                                 replace=False,    # sample without replacement to avoid duplication, overfitting and uniquness
                                 n_samples=125000, # to match malicious count
                                 random_state=42) 
# Downsample Malicious Label
data_malicious_downsampled = resample(data_malicious, 
                                 replace=False,    # sample without replacement to avoid duplication, overfitting and uniquness
                                 n_samples=125000, # to match benign count
                                 random_state=42) 
# Combine Benign and Malicious
sample_data = pd.concat([data_benign_downsampled, data_malicious_downsampled])

#Shuffle
sample_data = sample_data.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
sample_data['Label'].value_counts()

In [ ]:
os.makedirs("Data/Sample_Data", exist_ok=True) #creates a new directory for the sampled data
output_path = os.path.join("Data/Sample_Data", "sample_data.csv")
sample_data.to_csv(output_path, index=False)